# 02 - Preparação dos dados

Roda o ETL de `src/etl/` passo a passo e confere cada saída. O mesmo fluxo roda de uma vez com
`python -m src.etl.run_pipeline`.

In [1]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)
sns.set_theme(style="whitegrid")

from src.config import MIN_MESES_HISTORICO, MAX_MESES_INTERPOLACAO, HORIZONTE
from src.etl.extract import carregar_bruto
from src.etl.transform import padronizar, validar_linhas, regularizar_grade, resumir_itens, motivo_exclusao_modelagem
from src.features.build_features import construir_base, linhas_utilizaveis, FEATURES_NUMERICAS

## 1. Padronização e validação

In [2]:
df = padronizar(carregar_bruto())
df, rejeitadas = validar_linhas(df)
print("Linhas válidas:", len(df), "| rejeitadas:", len(rejeitadas))
df.head(3)

15:01:27 | INFO | src.etl.extract | Bruto: 8869 linhas, 74 séries


Linhas válidas: 8869 | rejeitadas: 0


,data,item,unidade,categoria_original,preco,serie_id,categoria
0,2015-01-01,All Ham (Excluding Canned Ham and Luncheon Sli...,lb. (453.6 gm),"Meat, broad category",3.211,APU0000FD2101,Suínos e frios
1,2015-02-01,All Ham (Excluding Canned Ham and Luncheon Sli...,lb. (453.6 gm),"Meat, broad category",3.221,APU0000FD2101,Suínos e frios
2,2015-03-01,All Ham (Excluding Canned Ham and Luncheon Sli...,lb. (453.6 gm),"Meat, broad category",3.176,APU0000FD2101,Suínos e frios


In [3]:
# conferindo as correções de categoria
df.loc[df["categoria"] != df["categoria_original"].map({
    "Bakery and grains": "Padaria e grãos", "Beef": "Carne bovina", "Dairy and fats": "Laticínios",
    "Drinks": "Bebidas", "Eggs": "Ovos", "Energy": "Energia", "Fruit": "Frutas",
    "Pantry and snacks": "Mercearia", "Pork and deli": "Suínos e frios", "Poultry": "Aves", "Vegetables": "Hortaliças",
}), ["item", "categoria_original", "categoria"]].drop_duplicates()

,item,categoria_original,categoria
0,All Ham (Excluding Canned Ham and Luncheon Sli...,"Meat, broad category",Suínos e frios
407,All Uncooked Beef Roasts,"Meat, broad category",Carne bovina
683,All Uncooked Other Beef (Excluding Veal),"Meat, broad category",Carne bovina
3462,"Coffee, 100%, ground roast, all sizes",Beef,Mercearia
6925,Potato chips,Vegetables,Mercearia


## 2. Grade mensal

Cada série vira uma sequência mensal contínua do primeiro ao último mês publicado. Buracos de até
2 meses são interpolados em escala log (a média geométrica respeita melhor variações percentuais);
maiores ficam vazios.

In [4]:
precos = regularizar_grade(df)
precos["imputado"].groupby(precos["data"]).sum().loc[lambda s: s > 0].sort_values(ascending=False).head(8)

15:01:27 | INFO | src.etl.transform | Grade mensal: 9324 linhas | 101 interpoladas | 354 ainda vazias


data
2025-10-01    60
2020-04-01    11
2025-11-01     4
2020-05-01     3
2016-09-01     2
2018-11-01     2
2020-03-01     2
2026-02-01     2
Name: imputado, dtype: int64

In [5]:
# exemplo: ovos em torno do shutdown
precos[(precos["item"].str.startswith("Eggs")) & (precos["data"].between("2025-08-01", "2026-01-01"))][["data", "preco", "imputado"]]

,data,preco,imputado
4054,2025-08-01,3.587,False
4055,2025-09-01,3.488,False
4056,2025-10-01,3.158,True
4057,2025-11-01,2.860,False
4058,2025-12-01,2.712,False
4059,2026-01-01,2.577,False


## 3. Tabela de itens e filtro da modelagem

In [6]:
itens = resumir_itens(precos)
itens["motivo_exclusao"] = motivo_exclusao_modelagem(itens)
print("Na modelagem:", itens["motivo_exclusao"].isna().sum(), "de", len(itens))
itens[itens["motivo_exclusao"].notna()][["item", "fim", "meses_publicados", "meses_faltando", "motivo_exclusao"]].sort_values("motivo_exclusao")

Na modelagem: 60 de 74


,item,fim,meses_publicados,meses_faltando,motivo_exclusao
9,"Chuck roast, graded and ungraded, excluding US...",2017-12-01,36,0,descontinuada
14,"Steak, round, graded and ungraded, excluding U...",2026-04-01,67,69,descontinuada
20,"Frankfurters, all meat or all beef",2022-04-01,79,0,descontinuada
21,"Bologna, all beef or mixed",2019-09-01,57,0,descontinuada
24,"Turkey, frozen, whole",2020-02-01,62,0,descontinuada
34,"Pears, Anjou",2019-07-01,31,7,descontinuada
36,"Grapes, Thompson Seedless",2021-09-01,77,4,descontinuada
40,"Peppers, sweet",2020-03-01,58,5,descontinuada
41,Broccoli,2020-03-01,63,0,descontinuada
46,"Sugar, white, 33-80 oz. pkg",2022-10-01,70,24,descontinuada


## 4. Base de modelagem

Uma linha por (série, mês-base t). O alvo é `log(preço em t+3) - log(preço em t)`.

| Feature | O que é |
|---|---|
| `ret_1`, `ret_3`, `ret_6`, `ret_12` | variação (log) nos últimos 1, 3, 6 e 12 meses |
| `dist_media_12` | quão acima/abaixo da média dos últimos 12 meses o preço está |
| `vol_6` | desvio-padrão das variações mensais nos últimos 6 meses |
| `sazonal_ano_anterior` | variação que aconteceu na mesma janela do ano anterior |
| `ret_3_categoria` | variação média de 3 meses da categoria no mesmo mês |
| `gasolina_ret_3`, `gasolina_ret_12` | variação recente da gasolina comum |
| `mes_alvo_sin`, `mes_alvo_cos` | mês do alvo em forma cíclica |
| `categoria` | one-hot |

In [7]:
series_ok = itens.loc[itens["motivo_exclusao"].isna(), "serie_id"]
base = construir_base(precos, series_ok)
uteis = linhas_utilizaveis(base)
print("Linhas na base:", len(base), "| com features e alvo completos:", len(uteis))
uteis[["item", "data_base", "data_alvo", "preco", "preco_alvo", "alvo"] + FEATURES_NUMERICAS[:4]].head()

15:01:28 | INFO | src.features.build_features | Base de modelagem: 8140 linhas, 60 séries, horizonte 3 meses


Linhas na base: 8140 | com features e alvo completos: 6975


,item,data_base,data_alvo,preco,preco_alvo,alvo,ret_1,ret_3,ret_6,ret_12
12,All Ham (Excluding Canned Ham and Luncheon Sli...,2016-01-01,2016-04-01,2.937,2.882,-0.018904,0.012333,-0.071600,-0.031504,-0.089194
13,All Ham (Excluding Canned Ham and Luncheon Sli...,2016-02-01,2016-05-01,2.928,2.909,-0.006510,-0.003069,-0.029280,-0.058050,-0.095372
14,All Ham (Excluding Canned Ham and Luncheon Sli...,2016-03-01,2016-06-01,2.950,3.076,0.041825,0.007486,0.016750,-0.057309,-0.073817
15,All Ham (Excluding Canned Ham and Luncheon Sli...,2016-04-01,2016-07-01,2.882,3.072,0.063844,-0.023321,-0.018904,-0.090504,-0.030414
16,All Ham (Excluding Canned Ham and Luncheon Sli...,2016-05-01,2016-08-01,2.909,3.098,0.062947,0.009325,-0.006510,-0.035790,-0.045692


In [8]:
# por que tantas linhas caem? principalmente os 12 primeiros meses de cada série (sem ret_12)
# e os 3 últimos (alvo ainda não existe)
base[FEATURES_NUMERICAS + ["alvo"]].isna().sum().sort_values(ascending=False).head(6)

ret_12                  855
sazonal_ano_anterior    824
dist_media_12           693
gasolina_ret_12         660
ret_6                   480
vol_6                   415
dtype: int64

## 5. Validações antes de salvar

In [9]:
assert base.duplicated(["serie_id", "data_base"]).sum() == 0
assert (uteis["preco_alvo"] > 0).all()
assert np.allclose(uteis["alvo"], np.log(uteis["preco_alvo"] / uteis["preco"]))
assert not uteis["alvo_imputado"].any(), "alvo interpolado não pode entrar em treino/teste"
print("ok")

ok


In [10]:
from src.etl import run_pipeline
run_pipeline.main()

15:01:28 | INFO | src.etl.extract | Bruto: 8869 linhas, 74 séries


15:01:29 | INFO | src.etl.transform | Grade mensal: 9324 linhas | 101 interpoladas | 354 ainda vazias


15:01:29 | INFO | src.etl.run_pipeline | Séries fora da modelagem: 14 de 74


15:01:29 | INFO | src.etl.run_pipeline |   descontinuada: 11


15:01:29 | INFO | src.etl.run_pipeline |   muitos_buracos: 3


15:01:29 | INFO | src.features.build_features | Base de modelagem: 8140 linhas, 60 séries, horizonte 3 meses


15:01:29 | INFO | src.etl.load | precos_mensais.parquet salvo (9324 linhas)


15:01:29 | INFO | src.etl.load | itens.parquet salvo (74 linhas)


15:01:29 | INFO | src.etl.load | series_rejeitadas.csv salvo (14 linhas)


15:01:29 | INFO | src.etl.load | base_modelagem.parquet salvo (8140 linhas)
